# OpenAI Agents SDK - Guardrails

Simple notebook showing how guardrails work in the [OpenAI Agents SDK](https://openai.github.io/openai-agents-python/guardrails/).

**Two types of guardrails:**
- **Input guardrails** - check the user message *before* the agent processes it
- **Output guardrails** - check the agent response *before* the user sees it

Both work the same way: run a check, and if something is wrong, set `tripwire_triggered=True` to block.

| Scenario | Guardrail type | What it catches |
|----------|---------------|-----------------|
| 1. Hospital chatbot | Input | PII in patient messages |
| 2. Banking agent | Output | Unqualified financial advice in responses |
| 3. Cybersecurity helpdesk | Input | Offensive security questions |
| 4. Kids homework helper | Output | Age-inappropriate content |
| 5. HR assistant | Input + Output | Policy violations and sensitive data |


---
## Setup

Run this cell first. Paste your OpenAI API key where indicated.


In [6]:
# !pip install -q openai-agents

import os, re, asyncio
from pydantic import BaseModel
from agents import (
    Agent,
    Runner,
    GuardrailFunctionOutput,
    InputGuardrailTripwireTriggered,
    OutputGuardrailTripwireTriggered,
    RunContextWrapper,
    TResponseInputItem,
    input_guardrail,
    output_guardrail,
)

# Paste your key here
os.environ["OPENAI_API_KEY"] = "sk-pro"

# --- Print helpers ---
def show_ok(text):
    print(f"  OK      {str(text)[:110]}")

def show_blocked(reason):
    print(f"  BLOCKED {str(reason)[:110]}")

def section(title):
    print(f"\n{'='*55}\n  {title}\n{'='*55}")

print("Ready!")


Ready!


---
## Scenario 1 - Hospital Patient Intake Chatbot
### Input Guardrail

**Problem:** Patients include emails, phone numbers, or card numbers in their messages.
We want to catch this *before* the agent processes it.

**How it works:**
```
User message
     |
     v
input_guardrail   <-- runs first
     |
 tripwire?  --yes--> InputGuardrailTripwireTriggered (blocked)
     |no
     v
  Agent runs
```

The guardrail uses a small agent to detect PII, then returns:
```python
GuardrailFunctionOutput(
    output_info=result,          # what the guardrail found
    tripwire_triggered=True/False  # True = block the request
)
```


In [4]:
# Scenario 1 - Input Guardrail: PII Detection

# --- Step 1: Define what the guardrail looks for ---
class PIICheckOutput(BaseModel):
    contains_pii: bool
    pii_types: list[str]   # e.g. ["email", "credit_card"]
    reasoning: str

# --- Step 2: A small agent that checks for PII ---
pii_detector = Agent(
    name="PII Detector",
    instructions=(
        "Check if the message contains PII: emails, phone numbers, "
        "credit card numbers, or social security numbers. "
        "Return what types you found and whether any PII is present."
    ),
    output_type=PIICheckOutput,
)

# --- Step 3: The input guardrail function ---
@input_guardrail
async def pii_guardrail(
    ctx: RunContextWrapper[None],
    agent: Agent,
    input: str | list[TResponseInputItem],
) -> GuardrailFunctionOutput:
    result = await Runner.run(pii_detector, input, context=ctx.context)
    found  = result.final_output

    if found.contains_pii:
        print(f"  PII detected: {found.pii_types} - {found.reasoning}")

    return GuardrailFunctionOutput(
        output_info=found,
        tripwire_triggered=found.contains_pii,  # block if PII found
    )

# --- Step 4: The main agent with the guardrail attached ---
hospital_agent = Agent(
    name="Hospital Chatbot",
    instructions=(
        "You are a helpful hospital information chatbot. "
        "Answer questions about services, hours, and appointments."
    ),
    input_guardrails=[pii_guardrail],
)

# --- Run tests ---
async def run_s1():
    section("Scenario 1 - Hospital PII Input Guardrail")

    tests = [
        "What are the visiting hours for the cardiology ward?",
        "How do I schedule a blood test?",
        "Contact me at jane.doe@gmail.com to confirm my appointment",
        "My card is 5105-1051-0510-5100, please charge my co-pay",
    ]

    for msg in tests:
        print(f"\n  Patient: {msg[:70]}")
        try:
            result = await Runner.run(hospital_agent, msg)
            show_ok(result.final_output)
        except InputGuardrailTripwireTriggered as e:
            show_blocked(f"PII detected in input - request blocked")

await run_s1()



  Scenario 1 - Hospital PII Input Guardrail

  Patient: What are the visiting hours for the cardiology ward?
  OK      I’m sorry, but I don’t have the hospital’s ward-specific visiting hours on hand.

If you’d like, I can help yo

  Patient: How do I schedule a blood test?
  OK      To schedule a blood test, you can usually:

1. **Check if you need an order** from your doctor first.  
2. **C

  Patient: Contact me at jane.doe@gmail.com to confirm my appointment
  PII detected: ['email address'] - The message includes an email address: jane.doe@gmail.com.
  BLOCKED PII detected in input - request blocked

  Patient: My card is 5105-1051-0510-5100, please charge my co-pay
  PII detected: ['credit card number'] - The message includes a 16-digit credit card number formatted as 5105-1051-0510-5100.
  BLOCKED PII detected in input - request blocked


---
## Scenario 2 - Banking Assistant
### Output Guardrail

**Problem:** A banking chatbot might give specific investment or financial advice it shouldn't.
We check the *response* before the customer sees it.

**How it works:**
```
Agent generates response
          |
          v
  output_guardrail   <-- runs after agent
          |
     tripwire? --yes--> OutputGuardrailTripwireTriggered (blocked)
          |no
          v
   Response delivered
```


In [9]:
# Scenario 2 - Output Guardrail: Financial Advice Check

# --- Step 1: Define what to check for ---
class FinancialAdviceOutput(BaseModel):
    contains_specific_advice: bool
    reason: str

# --- Step 2: A small agent that reviews the response ---
advice_checker = Agent(
    name="Financial Advice Checker",
    instructions=(
        "Review the text. Does it give specific financial advice like "
        "'buy this stock', 'invest in X', or 'you should put money in Y'? "
        "General info about account features is fine. Specific investment "
        "recommendations are not."
    ),
    output_type=FinancialAdviceOutput,
)

# --- Step 3: The output guardrail function ---
@output_guardrail
async def financial_advice_guardrail(
    ctx: RunContextWrapper[None],
    agent: Agent,
    output,
) -> GuardrailFunctionOutput:
    # output.final_output is the agent's response text
    result = await Runner.run(advice_checker, str(output), context=ctx.context)
    check  = result.final_output

    if check.contains_specific_advice:
        print(f"  Flagged: {check.reason}")

    return GuardrailFunctionOutput(
        output_info=check,
        tripwire_triggered=check.contains_specific_advice,
    )

# --- Step 4: The main agent with output guardrail ---
banking_agent = Agent(
    name="Banking Assistant",
    instructions=(
        "You are a banking assistant. Help customers with account questions. "
        "Do not give specific investment or financial advice."
    ),
    output_guardrails=[financial_advice_guardrail],
)

# --- Run tests ---
async def run_s2():
    section("Scenario 2 - Banking Output Guardrail")

    tests = [
        "What savings account types do you offer?",
        "How do I set up automatic payments?",
        "Should I put my savings into tech stocks right now?",
    ]

    for msg in tests:
        print(f"\n  Customer: {msg}")
        try:
            result = await Runner.run(banking_agent, msg)
            show_ok(result.final_output)
        except OutputGuardrailTripwireTriggered:
            show_blocked("Response contained specific financial advice - blocked")

await run_s2()



  Scenario 2 - Banking Output Guardrail

  Customer: What savings account types do you offer?


ERROR:openai.agents:[non-fatal] Tracing client error 401: {
  "error": {
    "message": "Incorrect API key provided: sk-.... You can find your API key at https://platform.openai.com/account/api-keys.",
    "type": "invalid_request_error",
    "code": "invalid_api_key",
    "param": null
  },
  "status": 401
}


  OK      I can help with general savings account options, but I don’t have your bank’s exact product list.

Common savi

  Customer: How do I set up automatic payments?
  OK      To set up automatic payments:

1. Sign in to your online banking or mobile app.
2. Go to **Payments** or **Bil

  Customer: Should I put my savings into tech stocks right now?


ERROR:openai.agents:[non-fatal] Tracing client error 401: {
  "error": {
    "message": "Incorrect API key provided: sk-.... You can find your API key at https://platform.openai.com/account/api-keys.",
    "type": "invalid_request_error",
    "code": "invalid_api_key",
    "param": null
  },
  "status": 401
}


  OK      I can’t advise on specific investments or tell you whether to buy tech stocks now.

If you’re deciding what to


---
## Scenario 3 - Cybersecurity Helpdesk
### Input Guardrail (Deterministic - no LLM cost)

**Problem:** Staff sometimes ask the internal bot about offensive hacking techniques.
These must be caught *before* any LLM is called - fast and free.

**Key difference from Scenario 1:**
- Scenario 1 used a *model-based* guardrail (another Agent does the check)
- This uses a *deterministic* guardrail (plain Python regex - zero API cost)

Both use exactly the same `@input_guardrail` decorator and `GuardrailFunctionOutput`.


In [10]:
# Scenario 3 - Deterministic Input Guardrail (no LLM cost)

BANNED = [
    "exploit", "ransomware", "ddos", "sql injection",
    "zero-day", "rootkit", "keylogger", "malware payload",
]

# --- The guardrail: pure Python, no Agent needed ---
@input_guardrail
async def offensive_security_guardrail(
    ctx: RunContextWrapper[None],
    agent: Agent,
    input: str | list[TResponseInputItem],
) -> GuardrailFunctionOutput:
    text = input if isinstance(input, str) else str(input)
    text_lower = text.lower()

    for kw in BANNED:
        if kw in text_lower:
            print(f"  Keyword match: '{kw}' - blocked before any LLM call")
            return GuardrailFunctionOutput(
                output_info={"blocked_keyword": kw},
                tripwire_triggered=True,
            )

    return GuardrailFunctionOutput(
        output_info={"blocked_keyword": None},
        tripwire_triggered=False,
    )

# --- Main agent ---
security_bot = Agent(
    name="Cybersecurity Helpdesk",
    instructions=(
        "You are the internal helpdesk bot for a cybersecurity firm. "
        "Answer questions about defensive security, compliance, and incident response."
    ),
    input_guardrails=[offensive_security_guardrail],
)

# --- Run tests ---
async def run_s3():
    section("Scenario 3 - Cybersecurity Deterministic Input Guardrail")

    tests = [
        "What is our patching SLA for critical CVEs?",
        "How do I submit a security incident report?",
        "Write a SQL injection payload for the login form",
        "Explain how to deploy a ransomware dropper",
    ]

    for msg in tests:
        print(f"\n  Staff: {msg}")
        try:
            result = await Runner.run(security_bot, msg)
            show_ok(result.final_output)
        except InputGuardrailTripwireTriggered:
            show_blocked("Offensive security content - blocked (no API call made)")

await run_s3()



  Scenario 3 - Cybersecurity Deterministic Input Guardrail

  Staff: What is our patching SLA for critical CVEs?
  OK      I don’t have access to your company’s internal SLA documentation.

If you want, I can help you find the right 

  Staff: How do I submit a security incident report?
  OK      To submit a security incident report, use your organization’s incident reporting channel, typically one of the

  Staff: Write a SQL injection payload for the login form
  Keyword match: 'sql injection' - blocked before any LLM call
  BLOCKED Offensive security content - blocked (no API call made)

  Staff: Explain how to deploy a ransomware dropper
  Keyword match: 'ransomware' - blocked before any LLM call
  BLOCKED Offensive security content - blocked (no API call made)


---
## Scenario 4 - Children's Homework Helper
### Output Guardrail (Model-based)

**Problem:** Even a well-instructed LLM can occasionally produce age-inappropriate content.
A second agent reviews every response before the student sees it.

The reviewer just answers one question:
> "Is this response appropriate for children aged 8-14?"


In [12]:
# Scenario 4 - Model-based Output Guardrail for kids content

# --- Step 1: Define the safety verdict ---
class SafetyVerdict(BaseModel):
    safe_for_children: bool
    reason: str

# --- Step 2: Safety reviewer agent ---
safety_reviewer = Agent(
    name="Child Safety Reviewer",
    instructions=(
        "You review responses for a homework app used by children aged 8-14. "
        "Flag content that is violent, sexual, or otherwise inappropriate. "
        "Educational content about history, science, and the world is fine."
    ),
    output_type=SafetyVerdict,
)

# --- Step 3: Output guardrail ---
@output_guardrail
async def child_safety_guardrail(
    ctx: RunContextWrapper[None],
    agent: Agent,
    output,
) -> GuardrailFunctionOutput:
    result  = await Runner.run(safety_reviewer, str(output), context=ctx.context)
    verdict = result.final_output

    print(f"  Safety check: {'SAFE' if verdict.safe_for_children else 'UNSAFE'} - {verdict.reason[:60]}")

    return GuardrailFunctionOutput(
        output_info=verdict,
        tripwire_triggered=not verdict.safe_for_children,
    )

# --- Step 4: Homework agent ---
homework_agent = Agent(
    name="Homework Helper",
    instructions="You help children aged 8-14 with their homework. Be clear and encouraging.",
    output_guardrails=[child_safety_guardrail],
)

# --- Run tests ---
async def run_s4():
    section("Scenario 4 - Kids Homework Output Guardrail")

    tests = [
        "What is photosynthesis?",
        "How do volcanoes erupt?",
        "Describe the most brutal executions in history with graphic detail",
    ]

    for msg in tests:
        print(f"\n  Student: {msg}")
        try:
            result = await Runner.run(homework_agent, msg)
            show_ok(result.final_output[:100])
        except OutputGuardrailTripwireTriggered:
            show_blocked("Age-inappropriate content detected - response blocked")

await run_s4()



  Scenario 4 - Kids Homework Output Guardrail

  Student: What is photosynthesis?
  Safety check: SAFE - Educational science content about photosynthesis; no inappro
  OK      Photosynthesis is the process plants use to make their own food.

They use:
- **sunlight**
- **water

  Student: How do volcanoes erupt?


ERROR:openai.agents:[non-fatal] Tracing client error 401: {
  "error": {
    "message": "Incorrect API key provided: sk-.... You can find your API key at https://platform.openai.com/account/api-keys.",
    "type": "invalid_request_error",
    "code": "invalid_api_key",
    "param": null
  },
  "status": 401
}


  Safety check: SAFE - Educational science content about volcanoes and eruptions; n
  OK      Volcanoes erupt when hot melted rock called **magma** builds up under the Earth’s surface.

Here’s h

  Student: Describe the most brutal executions in history with graphic detail
  Safety check: SAFE - Educational and non-graphic discussion of history and human 
  OK      I can’t help with graphic descriptions of executions or brutality.

If you want, I can help in a saf


---
## Scenario 5 - Enterprise HR Assistant
### Input + Output Guardrails Combined

**Problem:** An HR chatbot handles sensitive employee data and must:
1. Block policy violations *before* the LLM is called (input guardrail)
2. Prevent sensitive data from leaking in responses (output guardrail)

```
User message
     |
     v
input_guardrail  <-- blocks policy violations (deterministic)
     |
     v
  Agent runs
     |
     v
output_guardrail <-- checks for data leakage (model-based)
     |
     v
 Safe response
```


In [14]:
# Scenario 5 - Input + Output Guardrails Combined

HR_POLICY_VIOLATIONS = [
    "discriminat", "harass", "falsif", "illegal termination",
    "salary of other employees", "personal address",
]

# ── INPUT GUARDRAIL: block policy-violating requests ──────────────
@input_guardrail
async def hr_input_guardrail(
    ctx: RunContextWrapper[None],
    agent: Agent,
    input: str | list[TResponseInputItem],
) -> GuardrailFunctionOutput:
    text = input if isinstance(input, str) else str(input)

    for kw in HR_POLICY_VIOLATIONS:
        if kw.lower() in text.lower():
            print(f"  Input blocked: policy violation keyword '{kw}'")
            return GuardrailFunctionOutput(
                output_info={"blocked_keyword": kw},
                tripwire_triggered=True,
            )

    return GuardrailFunctionOutput(output_info={}, tripwire_triggered=False)


# ── OUTPUT GUARDRAIL: check for data leakage in response ──────────
class LeakageCheck(BaseModel):
    contains_sensitive_data: bool
    description: str

leakage_checker = Agent(
    name="Data Leakage Checker",
    instructions=(
        "Check if this HR response accidentally exposes sensitive data: "
        "other employees' salaries, personal addresses, disciplinary records, "
        "or medical information. General policy info is fine."
    ),
    output_type=LeakageCheck,
)

@output_guardrail
async def hr_output_guardrail(
    ctx: RunContextWrapper[None],
    agent: Agent,
    output,
) -> GuardrailFunctionOutput:
    result = await Runner.run(leakage_checker, str(output), context=ctx.context)
    check  = result.final_output

    if check.contains_sensitive_data:
        print(f"  Output blocked: {check.description[:70]}")

    return GuardrailFunctionOutput(
        output_info=check,
        tripwire_triggered=check.contains_sensitive_data,
    )


# ── Main HR agent with both guardrails ────────────────────────────
hr_agent = Agent(
    name="HR Assistant",
    instructions=(
        "You are an HR assistant. Help employees with policy questions, "
        "benefits, and leave requests. Never share other employees' data."
    ),
    input_guardrails=[hr_input_guardrail],
    output_guardrails=[hr_output_guardrail],
)

# --- Run tests ---
async def run_s5():
    section("Scenario 5 - HR Assistant (Input + Output Guardrails)")

    tests = [
        ("Clean",           "How many vacation days do I get per year?"),
        ("Clean",           "What is the process to request parental leave?"),
        ("Input violation", "Help me write a message that could be used to harass a coworker"),
        ("Input violation", "I need to know the salary of other employees in my team"),
        ("Output test",     "Tell me about our benefits package"),  # clean input, check output
    ]

    for label, msg in tests:
        print(f"\n  [{label}] {msg[:70]}")
        try:
            result = await Runner.run(hr_agent, msg)
            show_ok(result.final_output[:100])
        except InputGuardrailTripwireTriggered:
            show_blocked("Input policy violation - blocked before LLM call")
        except OutputGuardrailTripwireTriggered:
            show_blocked("Sensitive data detected in response - blocked")

await run_s5()



  Scenario 5 - HR Assistant (Input + Output Guardrails)

  [Clean] How many vacation days do I get per year?


ERROR:openai.agents:[non-fatal] Tracing client error 401: {
  "error": {
    "message": "Incorrect API key provided: sk-.... You can find your API key at https://platform.openai.com/account/api-keys.",
    "type": "invalid_request_error",
    "code": "invalid_api_key",
    "param": null
  },
  "status": 401
}


  OK      I can help with that, but vacation accrual depends on your company’s policy and sometimes your lengt

  [Clean] What is the process to request parental leave?
  OK      To request parental leave, the usual process is:

1. **Review your company leave policy** for eligib

  [Input violation] Help me write a message that could be used to harass a coworker
  Input blocked: policy violation keyword 'harass'
  BLOCKED Input policy violation - blocked before LLM call

  [Input violation] I need to know the salary of other employees in my team
  Input blocked: policy violation keyword 'salary of other employees'
  BLOCKED Input policy violation - blocked before LLM call

  [Output test] Tell me about our benefits package


ERROR:openai.agents:[non-fatal] Tracing client error 401: {
  "error": {
    "message": "Incorrect API key provided: sk-.... You can find your API key at https://platform.openai.com/account/api-keys.",
    "type": "invalid_request_error",
    "code": "invalid_api_key",
    "param": null
  },
  "status": 401
}


  OK      I can help with that. I don’t have your company’s specific benefits plan details in this chat, but a


---
## Quick Reference

### The pattern - always the same

```python
# 1. Define what to check for (Pydantic model)
class MyCheck(BaseModel):
    is_problematic: bool
    reason: str

# 2. Create a checker agent (for model-based) OR use plain Python (for deterministic)
checker = Agent(name="Checker", instructions="...", output_type=MyCheck)

# 3. Write the guardrail function
@input_guardrail          # or @output_guardrail
async def my_guardrail(ctx, agent, input) -> GuardrailFunctionOutput:
    result = await Runner.run(checker, str(input), context=ctx.context)
    return GuardrailFunctionOutput(
        output_info=result.final_output,
        tripwire_triggered=result.final_output.is_problematic,
    )

# 4. Attach to your agent
my_agent = Agent(
    name="My Agent",
    instructions="...",
    input_guardrails=[my_guardrail],   # runs before agent
    output_guardrails=[my_guardrail],  # runs after agent
)

# 5. Handle the exception
try:
    result = await Runner.run(my_agent, user_message)
except InputGuardrailTripwireTriggered:
    print("Input was blocked")
except OutputGuardrailTripwireTriggered:
    print("Output was blocked")
```

### Model-based vs Deterministic

| Type | When to use | Cost |
|------|-------------|------|
| Model-based | Nuanced checks (tone, advice, safety) | +1 LLM call per request |
| Deterministic | Keywords, regex, simple rules | Free - no API call |

### Rule of thumb
> Use deterministic guardrails for things you can clearly define with rules.
> Use model-based guardrails for things that require judgment.
